# FLUX ARC-AGI-3 Live Test — Full Cognitive System

**Testing FLUX on REAL ARC-AGI-3 games** with the complete cognitive stack:

| Component | Purpose |
|-----------|--------|
| **SpatialMemory** | Ice (curiosity) + Water (exploration mass) navigation |
| **CausalTracker** | Action → effect learning |
| **RuleInducer** | Pattern → rule abstraction |
| **GoalPlanner** | Objective decomposition |
| **PerceptionField** | Active vision with fovea |
| **GridToWave** | Grid → 432D wave encoding |

This notebook connects to the **real ARC-AGI-3 API** to test FLUX on games like:
- `ls20` — Agent reasoning
- `ft09` — Elementary logic
- `vc33` — Orchestration

---
## Cell 1: Environment Setup

In [ ]:
%%time
import os
import sys
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")
print(f"Root: {ROOT}")
os.chdir(ROOT)

# Add all phase paths
phase_paths = [
    ROOT,
    ROOT / 'phases/phase1',
    ROOT / 'phases/phase2',
    ROOT / 'phases/phase8',
    ROOT / 'phases/phase8_8',
    ROOT / 'phases/phase9_arc',    # Spatial Memory
    ROOT / 'phases/phase_unified', # Cognitive Layer
]
for p in phase_paths:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"\u2713 Environment configured with {len(phase_paths)} paths")

## Cell 2: Install Dependencies & Load API Key

In [ ]:
# Install arc-agi toolkit if not present
try:
    import arc_agi
    print("\u2713 arc-agi already installed")
except ImportError:
    print("Installing arc-agi...")
    !pip install -q arc-agi
    import arc_agi
    print("\u2713 arc-agi installed")

# ─────────────────────────────────────────────
# Load API Keys from Secrets (Kaggle/Colab first, then .env fallback)
# ─────────────────────────────────────────────

ARC_API_KEY = None
HF_TOKEN = None

# 1. Try Kaggle secrets first
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        ARC_API_KEY = secrets.get_secret('ARC_API_KEY')
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
        if ARC_API_KEY:
            print("\u2713 ARC_API_KEY loaded from Kaggle secrets")
        if HF_TOKEN:
            print("\u2713 HF_TOKEN loaded from Kaggle secrets")
    except Exception as e:
        print(f"\u26a0 Kaggle secrets error: {e}")

# 2. Try Colab secrets
elif IN_COLAB:
    try:
        from google.colab import userdata
        ARC_API_KEY = userdata.get('ARC_API_KEY')
        HF_TOKEN = userdata.get('HF_TOKEN')
        if ARC_API_KEY:
            print("\u2713 ARC_API_KEY loaded from Colab secrets")
        if HF_TOKEN:
            print("\u2713 HF_TOKEN loaded from Colab secrets")
    except Exception as e:
        print(f"\u26a0 Colab secrets error: {e}")

# 3. Try environment variables
if not ARC_API_KEY:
    ARC_API_KEY = os.environ.get('ARC_API_KEY')
    if ARC_API_KEY:
        print("\u2713 ARC_API_KEY loaded from environment")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("\u2713 HF_TOKEN loaded from environment")

# 4. Fallback to .env file (local development only)
if not ARC_API_KEY or not HF_TOKEN:
    env_file = ROOT / '.env'
    if env_file.exists():
        print("Loading from .env (local fallback)...")
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if not ARC_API_KEY and line.startswith('ARC_API_KEY='):
                    ARC_API_KEY = line.split('=', 1)[1]
                    print("\u2713 ARC_API_KEY loaded from .env")
                if not HF_TOKEN and line.startswith('hf_token='):
                    HF_TOKEN = line.split('=', 1)[1]
                    print("\u2713 HF_TOKEN loaded from .env")

# Set environment variables for libraries that need them
if ARC_API_KEY:
    os.environ['ARC_API_KEY'] = ARC_API_KEY
    print(f"  ARC_API_KEY: {ARC_API_KEY[:8]}...{ARC_API_KEY[-4:]}")
else:
    print("\u2717 ARC_API_KEY not found!")
    print("  Add to Kaggle/Colab secrets or set as environment variable")

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print(f"  HF_TOKEN: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
else:
    print("\u26a0 HF_TOKEN not found (model download may fail for private repos)")

## Cell 3: Device Setup & Load Model

In [ ]:
%%time
import torch
import numpy as np
from datetime import datetime

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load Flux-ARC.flx or download from HuggingFace
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

model_path = CHECKPOINTS_DIR / 'Flux-ARC.flx'

if not model_path.exists():
    print("Downloading Flux-ARC.flx from HuggingFace...")
    from huggingface_hub import hf_hub_download
    
    try:
        hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-ARC.flx',
            local_dir=str(ROOT),
            token=HF_TOKEN,  # Uses token from secrets loaded in Cell 2
        )
        print("\u2713 Downloaded Flux-ARC.flx")
    except Exception as e:
        print(f"\u26a0 Flux-ARC.flx download failed: {e}")
        print("  Trying Flux-Base.flx fallback...")
        try:
            hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename='checkpoints/Flux-Base.flx',
                local_dir=str(ROOT),
                token=HF_TOKEN,
            )
            model_path = CHECKPOINTS_DIR / 'Flux-Base.flx'
            print("\u2713 Downloaded Flux-Base.flx")
        except Exception as e2:
            print(f"\u2717 All downloads failed: {e2}")
            raise

# Load model
print(f"\nLoading {model_path.name}...")
flux_model = torch.load(str(model_path), map_location='cpu', weights_only=False)

print(f"\u2713 Loaded FLUX model")
print(f"  Format: {flux_model.get('format', 'unknown')}")
print(f"  Version: {flux_model.get('version', 'unknown')}")
print(f"  Components: {list(flux_model.get('components', {}).keys())[:8]}...")

## Cell 4: Initialize Cognitive Layer

In [ ]:
%%time
print("Initializing Cognitive Layer...")
print("=" * 60)

# Import cognitive components
from causal_tracker import CausalTracker
from rule_inducer import RuleInducer
from goal_planner import GoalPlanner, GoalType
from perception_field import PerceptionField
from arc_interface import GameAction, GameState, ActionCommand

# Initialize from model or fresh
if 'cognitive' in flux_model:
    print("Loading cognitive layer from checkpoint...")
    cog = flux_model['cognitive']
    
    ct_config = cog['causal_tracker']['config']
    causal_tracker = CausalTracker(
        max_history=ct_config.get('max_history', 1000),
        device='cpu',
    )
    causal_tracker.load_state_dict(cog['causal_tracker']['state_dict'])
    
    ri_config = cog['rule_inducer']['config']
    rule_inducer = RuleInducer(
        causal_tracker=causal_tracker,
        min_observations=ri_config.get('min_observations', 2),
        min_confidence=ri_config.get('min_confidence', 0.5),
    )
    rule_inducer.load_state_dict(cog['rule_inducer']['state_dict'])
    
    goal_planner = GoalPlanner()
    goal_planner.load_state_dict(cog['goal_planner']['state_dict'])
    
    pf_config = cog['perception_field']['config']
    perception_field = PerceptionField(
        max_size=pf_config.get('max_size', 64),
        fovea_radius=pf_config.get('fovea_radius', 5),
    )
    perception_field.load_state_dict(cog['perception_field']['state_dict'])
    
    print("\u2713 Loaded cognitive layer from Flux-ARC.flx")
else:
    print("Creating fresh cognitive layer...")
    causal_tracker = CausalTracker(max_history=1000, device='cpu')
    rule_inducer = RuleInducer(causal_tracker=causal_tracker)
    goal_planner = GoalPlanner()
    perception_field = PerceptionField(max_size=64, fovea_radius=5)
    print("\u2713 Created fresh cognitive layer")

print(f"\nCognitive Components Ready:")
print(f"  \u2713 CausalTracker (links: {len(causal_tracker.causal_links)})")
print(f"  \u2713 RuleInducer (rules: {len(rule_inducer.rules)})")
print(f"  \u2713 GoalPlanner")
print(f"  \u2713 PerceptionField (fovea_r={perception_field.fovea_radius})")

## Cell 5: Initialize Spatial Memory (Ice & Water)

In [ ]:
%%time
print("Initializing Spatial Memory (Ice & Water)...")
print("=" * 60)

from spatial_memory import SpatialMemory, NavigationTarget

# Create spatial memory instance
spatial_memory = SpatialMemory(
    max_size=64,          # ARC grids up to 64x64
    feature_dim=64,
    curiosity_threshold=0.1,
    device='cpu',
)

print("\u2713 SpatialMemory initialized")
print(f"  Max grid size: 64x64")
print(f"  Feature dim: 64")
print(f"  Curiosity threshold: 0.1")
print()
print("Dual-Field System:")
print("  \u2744 CURIOSITY ICE — Highlights interesting locations")
print("  \u223e EXPLORATION MASS — Tracks visited locations")

## Cell 6: Initialize Grid Encoder (GridToWave)

In [ ]:
%%time
print("Initializing GridToWave Encoder...")
print("=" * 60)

from grid_adapters import GridToWave, WaveToGrid

WAVE_DIM = 432

# Initialize encoder
grid_to_wave = GridToWave(wave_dim=WAVE_DIM, device='cpu')

# Try to load trained weights from model
if 'grid_adapters' in flux_model and 'encoder' in flux_model['grid_adapters']:
    grid_to_wave.load_state_dict(flux_model['grid_adapters']['encoder'])
    print("\u2713 Loaded trained GridToWave from checkpoint")
else:
    print("\u26a0 Using fresh GridToWave (no trained weights)")

grid_to_wave.eval()

# Test encoding
test_grid = [[0]*10 for _ in range(10)]
test_grid[5][5] = 1
with torch.no_grad():
    test_wave = grid_to_wave.encode(test_grid, mode='holistic')
    print(f"\nTest encoding: 10x10 grid -> [{WAVE_DIM}] wave")
    print(f"  Wave norm: {test_wave.norm().item():.4f}")

## Cell 7: Define FLUX-ARC Agent

In [ ]:
import torch.nn.functional as F
from collections import deque
import random

class FLUXARCAgent:
    """
    FLUX Agent for ARC-AGI-3 with full cognitive stack:
    - Spatial Memory (Ice & Water navigation)
    - CausalTracker (action → effect learning)
    - RuleInducer (pattern → rule abstraction)
    - GoalPlanner (objective decomposition)
    - PerceptionField (active vision)
    - GridToWave (grid encoding to 432D waves)
    """
    
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.reset()
        
    def reset(self):
        """Reset agent state for new game."""
        # Reset spatial memory
        spatial_memory.reset()
        
        # Reset perception
        perception_field.reset()
        
        # Agent state
        self.current_pos = (0, 0)
        self.current_target = None
        self.last_grid = None
        self.last_wave = None
        self.action_history = []
        self.observations = []
        self.step_count = 0
        
        # Performance tracking
        self.wins = 0
        self.total_actions = 0
        
    def log(self, msg: str):
        """Log message if verbose."""
        if self.verbose:
            print(f"  [FLUX] {msg}")
    
    def encode_grid(self, grid: list) -> torch.Tensor:
        """Encode grid to 432D wave using FLUX."""
        with torch.no_grad():
            wave = grid_to_wave.encode(grid, mode='holistic')
        return wave
    
    def find_agent_position(self, grid: list) -> tuple:
        """Find agent position in grid (heuristic: look for specific colors)."""
        # Common agent colors: 1 (red), 5 (gray)
        for r, row in enumerate(grid):
            for c, val in enumerate(row):
                if val == 1:  # Often the agent
                    return (r, c)
        return (0, 0)  # Default
    
    def find_target_cells(self, grid: list) -> list:
        """Find potential target cells (non-zero, non-agent)."""
        targets = []
        agent_pos = self.find_agent_position(grid)
        for r, row in enumerate(grid):
            for c, val in enumerate(row):
                if val > 0 and (r, c) != agent_pos:
                    targets.append((r, c, val))
        return targets
    
    def detect_changes(self, old_grid: list, new_grid: list) -> list:
        """Detect cell changes between grids."""
        changes = []
        if old_grid is None:
            return changes
        for r in range(min(len(old_grid), len(new_grid))):
            for c in range(min(len(old_grid[0]), len(new_grid[0]))):
                if old_grid[r][c] != new_grid[r][c]:
                    changes.append((r, c, old_grid[r][c], new_grid[r][c]))
        return changes
    
    def observe(self, grid: list, available_actions: list = None):
        """
        Process observation using full cognitive stack.
        
        1. Encode grid to wave
        2. Update spatial memory (ice & water)
        3. Update perception field
        4. Detect changes and record causality
        5. Update goal planner
        """
        self.step_count += 1
        grid_size = (len(grid), len(grid[0]) if grid else 0)
        
        # 1. Encode grid
        current_wave = self.encode_grid(grid)
        
        # 2. Find agent position
        self.current_pos = self.find_agent_position(grid)
        
        # 3. Update spatial memory
        obs_result = spatial_memory.observe(
            position=self.current_pos,
            local_view=grid,
            global_grid=grid,
        )
        
        # 4. Find targets and add curiosity
        targets = self.find_target_cells(grid)
        for r, c, color in targets:
            if r < spatial_memory.max_size and c < spatial_memory.max_size:
                spatial_memory.curiosity_field[r, c] += 2.0
        
        # 5. Detect changes → spawn ice
        changes = self.detect_changes(self.last_grid, grid)
        for r, c, old_val, new_val in changes:
            self.log(f"Change at ({r},{c}): {old_val} -> {new_val}")
            if r < spatial_memory.max_size and c < spatial_memory.max_size:
                spatial_memory.curiosity_field[r, c] += 5.0
            
            # Record causality if we took an action
            if self.action_history:
                last_action = self.action_history[-1]
                causal_tracker.record(
                    position=self.current_pos,
                    action=last_action,
                    grid_before=np.array(self.last_grid) if self.last_grid else np.zeros((1,1)),
                    grid_after=np.array(grid),
                )
        
        # 6. Update perception field
        perception_field.focus(self.current_pos, reason="agent")
        
        # 7. Decay fields
        spatial_memory.step_decay()
        
        # 8. Wave change detection
        wave_change = 0.0
        if self.last_wave is not None:
            cos_sim = F.cosine_similarity(
                current_wave.unsqueeze(0),
                self.last_wave.unsqueeze(0)
            ).item()
            wave_change = 1.0 - cos_sim
        
        # Store for next step
        self.last_grid = [row[:] for row in grid]
        self.last_wave = current_wave
        self.observations.append({
            'step': self.step_count,
            'pos': self.current_pos,
            'wave_change': wave_change,
            'changes': len(changes),
        })
        
        return {
            'position': self.current_pos,
            'wave_change': wave_change,
            'changes': changes,
            'targets': targets,
        }
    
    def decide_action(self, grid: list, available_actions: list) -> int:
        """
        Decide next action using cognitive stack.
        
        Strategy:
        1. Check if current target reached
        2. Get new target from spatial memory (highest ice, lowest mass)
        3. Navigate toward target
        4. Use ACTION5 (interact) if stuck or at target
        """
        grid_size = (len(grid), len(grid[0]) if grid else 0)
        
        # Get available simple actions (1-4 for movement)
        move_actions = [a for a in available_actions if a in [1, 2, 3, 4]]
        has_interact = 5 in available_actions
        has_click = 6 in available_actions
        
        # Check if we need a new target
        needs_new_target = (
            self.current_target is None or
            self.current_pos == self.current_target.position
        )
        
        if needs_new_target:
            self.current_target = spatial_memory.get_navigation_target(
                self.current_pos,
                grid_size,
            )
            if self.current_target:
                self.log(f"New target: {self.current_target.position} ({self.current_target.reason})")
        
        # If at target and can interact, do it
        if self.current_target and self.current_pos == self.current_target.position:
            if has_interact:
                self.log("At target, using ACTION5 (interact)")
                return 5
        
        # Navigate toward target
        if self.current_target and move_actions:
            action = spatial_memory.get_next_action(
                self.current_pos,
                self.current_target,
            )
            # Map spatial action (0-3) to ARC action (1-4)
            arc_action = action + 1
            if arc_action in move_actions:
                return arc_action
        
        # Fallback: random available action
        if move_actions:
            return random.choice(move_actions)
        if has_interact:
            return 5
        
        return available_actions[0] if available_actions else 1
    
    def get_action_with_data(self, grid: list, available_actions: list) -> tuple:
        """
        Get action and any required data (x,y for ACTION6).
        Returns (action_id, data_dict)
        """
        action = self.decide_action(grid, available_actions)
        data = {}
        
        # ACTION6 needs coordinates
        if action == 6:
            # Click on highest curiosity cell
            ice = spatial_memory.curiosity_field[:len(grid), :len(grid[0])]
            flat_idx = ice.argmax().item()
            y, x = divmod(flat_idx, len(grid[0]))
            data = {'x': x, 'y': y}
            self.log(f"ACTION6 click at ({x}, {y})")
        
        self.action_history.append(action)
        self.total_actions += 1
        
        return action, data
    
    def get_stats(self) -> dict:
        """Get agent performance stats."""
        grid_size = (64, 64)
        sm_stats = spatial_memory.get_stats(grid_size)
        return {
            'steps': self.step_count,
            'total_actions': self.total_actions,
            'wins': self.wins,
            'causal_links': len(causal_tracker.causal_links),
            'rules_induced': len(rule_inducer.rules),
            'coverage': sm_stats.get('coverage', 0),
        }

# Create agent
agent = FLUXARCAgent(verbose=True)
print("\u2713 FLUXARCAgent initialized with full cognitive stack")

## Cell 8: Test Connection to ARC API

In [ ]:
import requests

print("Testing ARC-AGI-3 API Connection...")
print("=" * 60)

ROOT_URL = "https://three.arcprize.org"

# Create session
session = requests.Session()
session.headers.update({
    "X-API-Key": ARC_API_KEY,
    "Accept": "application/json"
})

# Get games list
response = session.get(f"{ROOT_URL}/api/games")

if response.status_code == 200:
    games = response.json()
    print(f"\u2713 Connected! Found {len(games)} games")
    print(f"\nAvailable games:")
    for game in games[:10]:
        print(f"  - {game['game_id']}: {game.get('title', 'N/A')}")
    if len(games) > 10:
        print(f"  ... and {len(games) - 10} more")
else:
    print(f"\u2717 API Error: {response.status_code}")
    print(response.text)

## Cell 9: Play Single Game with FLUX Agent

In [ ]:
def play_game_with_flux(game_id: str, max_actions: int = 100, verbose: bool = True):
    """
    Play a single ARC-AGI-3 game using FLUX agent.
    
    Returns dict with results.
    """
    print(f"\n{'='*60}")
    print(f"Playing: {game_id}")
    print(f"{'='*60}")
    
    # Reset agent
    agent.reset()
    agent.verbose = verbose
    
    # Open scorecard
    response = session.post(
        f"{ROOT_URL}/api/scorecard/open",
        json={"tags": ["flux-test", game_id]}
    )
    if response.status_code != 200:
        print(f"\u2717 Failed to open scorecard: {response.text}")
        return None
    
    card_id = response.json()["card_id"]
    print(f"Scorecard: {card_id}")
    
    # Reset game
    response = session.post(
        f"{ROOT_URL}/api/cmd/RESET",
        json={"game_id": game_id, "card_id": card_id}
    )
    if response.status_code != 200:
        print(f"\u2717 RESET failed: {response.text}")
        return None
    
    game_data = response.json()
    guid = game_data["guid"]
    state = game_data["state"]
    score = game_data.get("score", 0)
    frame = game_data.get("frame", [[]])
    available_actions = game_data.get("available_actions", [1, 2, 3, 4, 5])
    
    print(f"Initial state: {state}, Score: {score}")
    print(f"Grid size: {len(frame)}x{len(frame[0]) if frame else 0}")
    print(f"Available actions: {available_actions}")
    
    # Play loop
    actions_taken = 0
    level_wins = 0
    
    while state == "NOT_FINISHED" and actions_taken < max_actions:
        # Observe
        obs = agent.observe(frame, available_actions)
        
        # Decide action
        action, action_data = agent.get_action_with_data(frame, available_actions)
        action_name = ['RESET', 'UP', 'DOWN', 'LEFT', 'RIGHT', 'INTERACT', 'CLICK', 'UNDO'][action]
        
        if verbose and actions_taken % 10 == 0:
            print(f"\nStep {actions_taken}: pos={agent.current_pos}, action={action_name}")
        
        # Build request
        request_data = {
            "game_id": game_id,
            "card_id": card_id,
            "guid": guid,
        }
        request_data.update(action_data)
        
        # Take action
        action_endpoint = f"ACTION{action}" if action > 0 else "RESET"
        response = session.post(
            f"{ROOT_URL}/api/cmd/{action_endpoint}",
            json=request_data
        )
        
        if response.status_code != 200:
            print(f"\u2717 Action failed: {response.text}")
            break
        
        game_data = response.json()
        state = game_data["state"]
        new_score = game_data.get("score", score)
        frame = game_data.get("frame", frame)
        available_actions = game_data.get("available_actions", available_actions)
        
        # Track level wins
        if new_score > score:
            print(f"  \u2191 Level completed! Score: {score} -> {new_score}")
            level_wins += 1
        
        score = new_score
        actions_taken += 1
    
    # Close scorecard
    response = session.post(
        f"{ROOT_URL}/api/scorecard/close",
        json={"card_id": card_id}
    )
    
    # Results
    result = {
        'game_id': game_id,
        'card_id': card_id,
        'final_state': state,
        'final_score': score,
        'actions_taken': actions_taken,
        'level_wins': level_wins,
        'agent_stats': agent.get_stats(),
    }
    
    print(f"\n{'─'*40}")
    print(f"Result: {state}")
    print(f"Score: {score}")
    print(f"Actions: {actions_taken}")
    print(f"Levels won: {level_wins}")
    print(f"Scorecard: {ROOT_URL}/scorecards/{card_id}")
    
    return result

print("\u2713 play_game_with_flux() defined")

## Cell 10: Run FLUX on ls20

In [ ]:
%%time
# Test on ls20 - Agent Reasoning game
result_ls20 = play_game_with_flux("ls20", max_actions=50, verbose=True)

## Cell 11: Run FLUX on ft09

In [ ]:
%%time
# Test on ft09 - Elementary Logic game
result_ft09 = play_game_with_flux("ft09", max_actions=50, verbose=True)

## Cell 12: Run FLUX on vc33

In [ ]:
%%time
# Test on vc33 - Orchestration game
result_vc33 = play_game_with_flux("vc33", max_actions=50, verbose=True)

## Cell 13: Aggregate Results

In [ ]:
print("\n" + "=" * 60)
print("FLUX ARC-AGI-3 TEST RESULTS")
print("=" * 60)

results = [r for r in [result_ls20, result_ft09, result_vc33] if r is not None]

print(f"\nGames tested: {len(results)}")
print(f"\n{'Game':<15} {'State':<15} {'Score':<10} {'Actions':<10} {'Levels':<10}")
print("-" * 60)

total_score = 0
total_actions = 0
total_wins = 0

for r in results:
    print(f"{r['game_id']:<15} {r['final_state']:<15} {r['final_score']:<10} {r['actions_taken']:<10} {r['level_wins']:<10}")
    total_score += r['final_score']
    total_actions += r['actions_taken']
    total_wins += r['level_wins']

print("-" * 60)
print(f"{'TOTAL':<15} {'':<15} {total_score:<10} {total_actions:<10} {total_wins:<10}")

print(f"\n\nAgent Stats:")
stats = agent.get_stats()
for key, val in stats.items():
    print(f"  {key}: {val}")

print(f"\n\nCognitive Layer Stats:")
print(f"  Causal links learned: {len(causal_tracker.causal_links)}")
print(f"  Rules induced: {len(rule_inducer.rules)}")

## Cell 14: Visualize Spatial Memory

In [ ]:
import matplotlib.pyplot as plt

# Get the last grid size from agent
grid_size = 30  # Default
if agent.last_grid:
    grid_size = max(len(agent.last_grid), len(agent.last_grid[0]) if agent.last_grid else 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Exploration Mass (Water)
mass = spatial_memory.exploration_mass[:grid_size, :grid_size].cpu().numpy()
im1 = axes[0].imshow(mass, cmap='hot', vmin=0)
axes[0].set_title('Exploration Mass (Water)\nWhere FLUX has been')
plt.colorbar(im1, ax=axes[0])

# Curiosity Ice
ice = spatial_memory.curiosity_field[:grid_size, :grid_size].cpu().numpy()
im2 = axes[1].imshow(ice, cmap='cool', vmin=0)
axes[1].set_title('Curiosity Ice\nWhat is interesting')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

# ASCII visualization
print("\nSpatial Memory ASCII View:")
print(spatial_memory.visualize((grid_size, grid_size), agent.current_pos))

## Cell 15: Summary

In [ ]:
print("=" * 60)
print("FLUX ARC-AGI-3 LIVE TEST COMPLETE")
print("=" * 60)

print(f"""
Model: {model_path.name}
Games tested: {len(results)}
Total levels won: {total_wins}
Total actions: {total_actions}

Cognitive Stack Used:
  \u2713 SpatialMemory (Ice & Water navigation)
  \u2713 CausalTracker ({len(causal_tracker.causal_links)} links learned)
  \u2713 RuleInducer ({len(rule_inducer.rules)} rules)
  \u2713 GoalPlanner
  \u2713 PerceptionField
  \u2713 GridToWave encoder

View scorecards at:
""")

for r in results:
    print(f"  {r['game_id']}: {ROOT_URL}/scorecards/{r['card_id']}")

print(f"""

Next Steps:
  1. Train GridToWave on more ARC patterns
  2. Improve rule induction from causal observations
  3. Test on more games
  4. Submit to competition (OperationMode.COMPETITION)
""")